# Axis 4: Efficiency Trade-offs

Research Question: Which quantization offers the best trade-off between
quality and efficiency for practical RAG deployment?

Evaluation Design:
- Latency: ms/token, tokens/sec, time-to-first-token
- Throughput: Tokens per second across batch sizes
- Memory: Allocated, reserved, peak, KV cache estimates
- Model Size: Disk size, compression ratio
- Quality-Efficiency Pareto: F1 vs speed, F1 vs memory

Setup and Installation

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

subprocess.run(['rm', '-rf', '/kaggle/working/qrag'], check=False)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/zahraamselim/qrag.git"
subprocess.run(['git', 'clone', repo_url, '/kaggle/working/qrag'], check=True)

sys.path.insert(0, '/kaggle/working/qrag')
os.chdir('/kaggle/working/qrag')

print("Repository cloned successfully")

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'transformers==4.51.3', 'accelerate', 'exllamav2',
    'autoawq', 'bitsandbytes', 'scipy', 'numpy', 'huggingface-hub'
], check=True)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("Dependencies installed")

Imports

In [ ]:
import gc
import torch
import logging
import numpy as np
import random

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

from utils import (
    save_json,
    load_model,
    cleanup_model,
    clear_memory
)
from metrics.efficiency import (
    measure_latency,
    measure_memory_usage,
    reset_memory_stats,
    get_model_size,
    compute_compression_ratio,
    compare_quantization_efficiency
)

print("Imports successful")

Configuration

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/efficiency_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

NUM_WARMUP = 3
NUM_RUNS = 10
MAX_NEW_TOKENS = 128

TEST_PROMPTS = [
    "The capital of France is",
    "Artificial intelligence is defined as",
    "In machine learning, overfitting occurs when",
    "The theory of relativity states that",
    "Natural language processing enables",
    "Quantum computing differs from classical computing by",
    "The human brain contains approximately",
    "Climate change is primarily caused by",
    "The internet was originally developed to",
    "DNA stands for deoxyribonucleic acid and"
]

all_results = {}

logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"Device: {DEVICE}")
logger.info(f"Warmup runs: {NUM_WARMUP}, Measurement runs: {NUM_RUNS}")

if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Evaluation Function

In [ ]:
def evaluate_efficiency(model_name, model_variant, quantization, backend='huggingface'):
    """
    Evaluate efficiency metrics for a single model configuration.
    
    Measures:
    - Latency (ms/token, tokens/sec, TTFT)
    - Memory (allocated, reserved, peak, KV cache)
    - Model size (GB, compression ratio)
    """
    config_name = f"{quantization}_{model_variant}"
    logger.info(f"EVALUATING EFFICIENCY: {config_name.upper()}")
    logger.info(f"Model: {model_name}")
    
    try:
        clear_memory()
        reset_memory_stats(DEVICE)
        
        model, tokenizer, extra_objects = load_model(model_name, quantization, backend)
        
        logger.info("\nMeasuring latency and throughput")
        timing = measure_latency(
            model=model,
            tokenizer=tokenizer,
            prompts=TEST_PROMPTS,
            num_warmup=NUM_WARMUP,
            num_runs=NUM_RUNS,
            max_new_tokens=MAX_NEW_TOKENS,
            backend=backend
        )
        
        logger.info("Measuring memory usage")
        if quantization == 'gptq' and extra_objects:
            memory = measure_memory_usage(device=DEVICE, model=model, config=extra_objects.get('config'))
        else:
            memory = measure_memory_usage(device=DEVICE, model=model)
        
        logger.info("Computing model size")
        if quantization in ['gptq', 'awq', 'nf4']:
            if quantization == 'gptq' and extra_objects:
                size = get_model_size(model=model, config=extra_objects.get('config'), bits=4)
            else:
                size = get_model_size(model=model, bits=4)
        else:
            size = get_model_size(model=model)
        
        results = {
            'model_name': model_name,
            'quantization': quantization,
            'variant': model_variant,
            'backend': backend,
            'timing': timing,
            'memory': memory,
            'model_size': size,
            'config': {
                'num_warmup': NUM_WARMUP,
                'num_runs': NUM_RUNS,
                'max_new_tokens': MAX_NEW_TOKENS,
                'num_test_prompts': len(TEST_PROMPTS),
                'random_seed': RANDOM_SEED
            }
        }
        
        logger.info("\nResults Summary:")
        if 'error' not in timing:
            logger.info(f"  ms/token: {timing['ms_per_token']['mean']:.2f} +/- {timing['ms_per_token']['std']:.2f}")
            logger.info(f"  tokens/sec: {timing['tokens_per_sec']['mean']:.2f}")
            logger.info(f"  TTFT: {timing.get('ttft_ms', 0):.2f} ms")
        
        if 'error' not in memory:
            logger.info(f"  Memory (peak): {memory['peak_mb']:.2f} MB")
            logger.info(f"  Memory (allocated): {memory['allocated_mb']:.2f} MB")
        
        if 'error' not in size:
            logger.info(f"  Model size: {size['size_gb']:.3f} GB")
            if 'bits_per_param' in size:
                logger.info(f"  Bits/param: {size['bits_per_param']:.1f}")
        
        save_json(results, OUTPUT_DIR / f'{config_name}.json')
        
        logger.info(f"COMPLETED: {config_name.upper()}")
        
        if backend == 'exllama' and extra_objects and 'cache' in extra_objects:
            extra_objects['cache'].current_seq_len = 0
        
        cleanup_model(model, tokenizer, extra_objects)
        clear_memory()
        
        return results
        
    except Exception as e:
        logger.error(f"Evaluation failed for {config_name}: {e}")
        import traceback
        traceback.print_exc()
        clear_memory()
        return {'error': str(e), 'config': config_name}

Model Evaluations

In [ ]:
logger.info("Starting FP16 Base evaluation")
all_results['fp16_base'] = evaluate_efficiency('mistralai/Mistral-7B-v0.1', 'base', 'fp16', 'huggingface')

In [ ]:
logger.info("Starting FP16 Instruct evaluation")
all_results['fp16_instruct'] = evaluate_efficiency('mistralai/Mistral-7B-Instruct-v0.2', 'instruct', 'fp16', 'huggingface')

In [ ]:
logger.info("Starting GPTQ Base evaluation")
all_results['gptq_base'] = evaluate_efficiency('TheBloke/Mistral-7B-v0.1-GPTQ', 'base', 'gptq', 'exllama')

In [ ]:
logger.info("Starting GPTQ Instruct evaluation")
all_results['gptq_instruct'] = evaluate_efficiency('TheBloke/Mistral-7B-Instruct-v0.2-GPTQ', 'instruct', 'gptq', 'exllama')

In [ ]:
logger.info("Starting AWQ Base evaluation")
all_results['awq_base'] = evaluate_efficiency('TheBloke/Mistral-7B-v0.1-AWQ', 'base', 'awq', 'huggingface')

In [ ]:
logger.info("Starting AWQ Instruct evaluation")
all_results['awq_instruct'] = evaluate_efficiency('TheBloke/Mistral-7B-Instruct-v0.2-AWQ', 'instruct', 'awq', 'huggingface')

In [ ]:
logger.info("Starting NF4 Base evaluation")
all_results['nf4_base'] = evaluate_efficiency('mistralai/Mistral-7B-v0.1', 'base', 'nf4', 'huggingface')

In [ ]:
logger.info("Starting NF4 Instruct evaluation")
all_results['nf4_instruct'] = evaluate_efficiency('mistralai/Mistral-7B-Instruct-v0.2', 'instruct', 'nf4', 'huggingface')

Comprehensive Analysis and Reporting

In [ ]:
save_json(all_results, OUTPUT_DIR / 'complete_efficiency_results.json')

logger.info("AXIS 4 EVALUATION COMPLETE: EFFICIENCY TRADE-OFFS")
logger.info(f"\nTotal configurations evaluated: {len(all_results)}")

logger.info("\nLatency and Throughput Summary")
logger.info(f"{'Config':<20} {'ms/token':<15} {'tokens/sec':<15} {'TTFT (ms)':<15}")

for config_name, config_data in sorted(all_results.items()):
    if 'error' in config_data:
        logger.info(f"{config_name:<20} ERROR")
        continue
    
    timing = config_data.get('timing', {})
    if 'error' in timing:
        logger.info(f"{config_name:<20} ERROR")
        continue
    
    ms_token = timing['ms_per_token']['mean']
    tps = timing['tokens_per_sec']['mean']
    ttft = timing.get('ttft_ms', 0)
    
    logger.info(f"{config_name:<20} {ms_token:<15.2f} {tps:<15.2f} {ttft:<15.2f}")

logger.info("\nMemory Usage Summary")
logger.info(f"{'Config':<20} {'Peak (MB)':<15} {'Allocated (MB)':<18} {'KV-512 (MB)':<15}")

for config_name, config_data in sorted(all_results.items()):
    if 'error' not in config_data:
        memory = config_data.get('memory', {})
        if 'error' not in memory:
            peak = memory.get('peak_mb', 0)
            allocated = memory.get('allocated_mb', 0)
            kv = memory.get('kv_cache_512', 0)
            
            logger.info(f"{config_name:<20} {peak:<15.2f} {allocated:<18.2f} {kv:<15.2f}")

logger.info("\nModel Size Summary")
logger.info(f"{'Config':<20} {'Size (GB)':<15} {'Bits/param':<15} {'Compression':<15}")

fp16_sizes = []
for name, data in all_results.items():
    if 'fp16' in name and 'error' not in data and 'error' not in data.get('model_size', {}):
        fp16_sizes.append(data['model_size']['size_gb'])

baseline_size = np.mean(fp16_sizes) if fp16_sizes else 0

for config_name, config_data in sorted(all_results.items()):
    if 'error' not in config_data:
        size = config_data.get('model_size', {})
        if 'error' not in size:
            size_gb = size.get('size_gb', 0)
            bits = size.get('bits_per_param', 0)
            
            if baseline_size > 0:
                compression = compute_compression_ratio(
                    int(baseline_size * 1024**3),
                    int(size_gb * 1024**3)
                )
                comp_str = f"{compression['ratio']:.2f}x"
            else:
                comp_str = "N/A"
            
            logger.info(f"{config_name:<20} {size_gb:<15.3f} {bits:<15.1f} {comp_str:<15}")

logger.info("\nQuantization Method Comparison")
quant_metrics = {}
for method in ['fp16', 'gptq', 'awq', 'nf4']:
    speeds = []
    memories = []
    sizes = []
    
    for name, data in all_results.items():
        if method in name and 'error' not in data:
            if 'error' not in data.get('timing', {}):
                speeds.append(data['timing']['tokens_per_sec']['mean'])
            if 'error' not in data.get('memory', {}):
                memories.append(data['memory']['peak_mb'])
            if 'error' not in data.get('model_size', {}):
                sizes.append(data['model_size']['size_gb'])
    
    if speeds:
        quant_metrics[method] = {
            'avg_speed': np.mean(speeds),
            'avg_memory': np.mean(memories) if memories else 0,
            'avg_size': np.mean(sizes) if sizes else 0
        }

logger.info(f"\n{'Method':<15} {'Tokens/sec':<15} {'Memory (MB)':<15} {'Size (GB)':<15}")
for method in sorted(quant_metrics.keys()):
    metrics = quant_metrics[method]
    logger.info(f"{method:<15} {metrics['avg_speed']:<15.2f} {metrics['avg_memory']:<15.2f} {metrics['avg_size']:<15.3f}")

logger.info("\nEfficiency Rankings")
if quant_metrics:
    speed_ranking = sorted(quant_metrics.items(), key=lambda x: x[1]['avg_speed'], reverse=True)
    logger.info("\nBy Speed (fastest first):")
    for i, (method, metrics) in enumerate(speed_ranking, 1):
        logger.info(f"  {i}. {method}: {metrics['avg_speed']:.2f} tokens/sec")
    
    memory_ranking = sorted(quant_metrics.items(), key=lambda x: x[1]['avg_memory'])
    logger.info("\nBy Memory (lowest first):")
    for i, (method, metrics) in enumerate(memory_ranking, 1):
        logger.info(f"  {i}. {method}: {metrics['avg_memory']:.2f} MB")
    
    size_ranking = sorted(quant_metrics.items(), key=lambda x: x[1]['avg_size'])
    logger.info("\nBy Model Size (smallest first):")
    for i, (method, metrics) in enumerate(size_ranking, 1):
        logger.info(f"  {i}. {method}: {metrics['avg_size']:.3f} GB")

logger.info("\nQuantization Efficiency Comparison")
fp16_base_metrics = all_results.get('fp16_base', {}).get('timing', {})
if 'error' not in fp16_base_metrics:
    logger.info("\nComparing quantized models to FP16 Base:")
    
    for config_name, config_data in sorted(all_results.items()):
        if config_name == 'fp16_base' or 'error' in config_data:
            continue
        
        timing = config_data.get('timing', {})
        memory = config_data.get('memory', {})
        
        if 'error' not in timing and 'error' not in memory:
            fp16_memory = all_results['fp16_base'].get('memory', {})
            
            comparison = compare_quantization_efficiency(
                {'mean_ms': fp16_base_metrics['ms_per_token']['mean'],
                 'mean_tps': fp16_base_metrics['tokens_per_sec']['mean'],
                 'allocated_mb': fp16_memory.get('allocated_mb', 0)},
                {'mean_ms': timing['ms_per_token']['mean'],
                 'mean_tps': timing['tokens_per_sec']['mean'],
                 'allocated_mb': memory.get('allocated_mb', 0)}
            )
            
            logger.info(f"\n{config_name}:")
            logger.info(f"  Speedup: {comparison['speedup']:.2f}x")
            logger.info(f"  Memory reduction: {comparison['memory_reduction_percent']:.1f}%")
            logger.info(f"  Throughput improvement: {comparison['throughput_improvement']:.2f}x")

logger.info(f"\nResults saved to: {OUTPUT_DIR}")
logger.info("Efficiency evaluation complete")